In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_absolute_error, accuracy_score

In [21]:
df = pd.read_csv('/Users/abhi/delay_aware/flight-delay-prediction/data/Airline_Delay_Cause.csv')

In [22]:
print(df.head())
print(df.info())
print('Total rows:', len(df))

   year  month carrier       carrier_name airport  \
0  2023      8      9E  Endeavor Air Inc.     ABE   
1  2023      8      9E  Endeavor Air Inc.     ABY   
2  2023      8      9E  Endeavor Air Inc.     AEX   
3  2023      8      9E  Endeavor Air Inc.     AGS   
4  2023      8      9E  Endeavor Air Inc.     ALB   

                                        airport_name  arr_flights  arr_del15  \
0  Allentown/Bethlehem/Easton, PA: Lehigh Valley ...         89.0       13.0   
1             Albany, GA: Southwest Georgia Regional         62.0       10.0   
2           Alexandria, LA: Alexandria International         62.0       10.0   
3        Augusta, GA: Augusta Regional at Bush Field         66.0       12.0   
4                   Albany, NY: Albany International         92.0       22.0   

   carrier_ct  weather_ct  ...  security_ct  late_aircraft_ct  arr_cancelled  \
0        2.25        1.60  ...          0.0              5.99            2.0   
1        1.97        0.04  ...          

In [23]:
df['Delayed'] = (df['arr_del15'] > 0).astype(int)  


In [24]:
cat_cols = ['carrier', 'airport']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [25]:
features = ['year', 'month', 'carrier', 'airport', 
            'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']

In [26]:
X = df[features]
y_class = df['Delayed']
y_reg = df['arr_delay']  # average delay in minutes

In [27]:
# Train-test split
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42)

In [28]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_class_train)
y_class_pred = clf.predict(X_test)

In [29]:
print("Classification Report:")
print(classification_report(y_class_test, y_class_pred))
print("Classification Accuracy:", accuracy_score(y_class_test, y_class_pred))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1378
           1       1.00      1.00      1.00     32956

    accuracy                           1.00     34334
   macro avg       1.00      1.00      1.00     34334
weighted avg       1.00      1.00      1.00     34334

Classification Accuracy: 1.0


In [30]:
# Regression model (predict delay minutes only for predicted delayed flights)
X_reg_train = X_train[y_class_train == 1]
y_reg_train_delayed = y_reg_train[y_class_train == 1]

In [31]:
X_reg_test = X_test[y_class_pred == 1]
y_reg_test_delayed = y_reg_test[y_class_pred == 1]


In [32]:
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_reg_train, y_reg_train_delayed)
y_reg_pred = reg.predict(X_reg_test)

In [33]:
print(f"Mean Absolute Error for delay minutes prediction: {mean_absolute_error(y_reg_test_delayed, y_reg_pred):.2f}")

Mean Absolute Error for delay minutes prediction: 94.24
